# Movie Ratings Analysis - Normalization and Standardization

Loading a CSV with movie ratings from friends and computing averages, normalized ratings, and standardized ratings.

In [ ]:
import pandas as pd
from pathlib import Path

## Step 1 - Load the CSV

Using index_col=0 so pandas treats the names column as the row index. encoding='utf-8-sig' handles a hidden character Excel sometimes adds. Path.cwd() builds the path relative to wherever this notebook is running from.

In [ ]:
CSV_PATH = Path.cwd() / "Movie_Ratings_by_Friends.csv"
df = pd.read_csv(CSV_PATH, index_col=0, encoding="utf-8-sig")
df

## Step 2 - Average ratings (original)

.mean() skips NaN values automatically. axis=1 gives one average per user, axis=0 gives one per movie.

In [ ]:
print("Average per user:")
print(df.mean(axis=1).round(2).to_frame(name="Avg Rating"))

print("\nAverage per movie:")
print(df.mean(axis=0).round(2).to_frame(name="Avg Rating"))

## Step 3 - Normalized ratings

Min-max normalization rescales each user's ratings to a 0-1 range. I'm doing it per user so that differences in how people use the scale don't skew the results. Added a check for when min equals max to avoid division by zero.

In [ ]:
def normalize_row(row):
    row_min = row.min()
    row_max = row.max()
    if row_max == row_min:
        return row.apply(lambda x: 0.5 if pd.notna(x) else float("nan"))
    return (row - row_min) / (row_max - row_min)

df_normalized = df.apply(normalize_row, axis=1)
df_normalized.round(4)

In [ ]:
print("Average per user (normalized):")
print(df_normalized.mean(axis=1).round(4).to_frame(name="Avg Normalized"))

print("\nAverage per movie (normalized):")
print(df_normalized.mean(axis=0).round(4).to_frame(name="Avg Normalized"))

## Step 4 - Pros and cons of normalized vs actual ratings

Advantages:
- Removes bias from how each person uses the scale
- Makes relative preferences easier to compare across users
- Useful for ML models that expect values in a fixed range

Disadvantages:
- Loses the actual rating values, so we can't tell if someone truly loved a movie
- One outlier rating shifts the whole scale for that user
- Less reliable for users with very few ratings

## Step 5 (Extra Credit) - Standardized ratings

Z-score standardization shifts each user's ratings so their mean is 0 and standard deviation is 1. A positive score means above that user's average, negative means below. Unlike normalization there is no fixed output range, which makes it less sensitive to outliers.

In [ ]:
def standardize_row(row):
    row_mean = row.mean()
    row_std = row.std()
    if pd.isna(row_std) or row_std == 0:
        return row.apply(lambda x: 0.0 if pd.notna(x) else float("nan"))
    return (row - row_mean) / row_std

df_standardized = df.apply(standardize_row, axis=1)
df_standardized.round(4)

In [ ]:
print("Average per user (standardized):")
print(df_standardized.mean(axis=1).round(4).to_frame(name="Avg Z-score"))

print("\nAverage per movie (standardized):")
print(df_standardized.mean(axis=0).round(4).to_frame(name="Avg Z-score"))

Every user's average Z-score is 0.0, which makes sense since we subtracted each person's own mean. For movies, a positive average means it was generally rated above each person's personal average.